# LLM Fine-Tuning for Indonesian Health Regulations

**Permenkes No. 10 Tahun 2024 - PEFT/QLoRA Fine-Tuning Pipeline**

This notebook contains the complete pipeline for:
1. **Data Preprocessing** - Extract and prepare training data from PDF
2. **Model Training** - Fine-tune LLaMA 3 8B with QLoRA
3. **Inference Demo** - Test the model with health regulation queries

**Target Environment:** Google Colab T4 GPU (16GB VRAM)

---
## Setup & Dependencies

In [1]:
# Clone repository
!git clone https://github.com/mpfordreamer/paperlesshospital-test.git
%cd paperlesshospital-test

fatal: destination path 'paperlesshospital-test' already exists and is not an empty directory.
/content/paperlesshospital-test


In [2]:
!pip install unsloth

In [3]:
# Install rouge score for evaluation
!pip install rouge_score

In [4]:
import re
import json
import random
from pathlib import Path

import torch
import pdfplumber
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/tmp/ipython-input-3635299683.py:10: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


---
## Configuration

In [5]:
# Paths
PDF_PATH = Path("data/raw/permenkes-no-10-tahun-2024.pdf")
DATASET_PATH = Path("data/dataset.jsonl")
OUTPUT_DIR = Path("outputs")


# Model
BASE_MODEL = "unsloth/llama-3-8b-bnb-4bit"

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training Hyperparameters (optimized for T4 GPU)
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 200
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

---
# Phase 1: Data Preprocessing

Extract text from PDF, parse articles (Pasal), and generate instruction-tuning dataset.

### 1.1 PDF Text Extraction & Cleaning

In [6]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract raw text from PDF using pdfplumber."""
    full_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n\n".join(full_text)


raw_text = extract_text_from_pdf(PDF_PATH)
print(f"Extracted {len(raw_text):,} characters")

Extracted 11,850 characters


In [7]:
def clean_text_robust(text: str) -> str:
    """Clean PDF text by removing noise."""
    # Remove page numbers (e.g., - 2 -)
    text = re.sub(r'\n\s*-\s*\d+\s*-\s*\n', '\n', text)

    # Remove signature block
    if "Ditetapkan di" in text:
        text = text.split("Ditetapkan di")[0]

    # Flatten whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 1.2 Article Pasal Parsing

In [8]:
def parse_articles_fixed(text: str) -> list:
    """Parse articles, skipping preamble section."""

    # Skip preamble (Menimbang/Mengingat) - start after MEMUTUSKAN
    if "MEMUTUSKAN" in text:
        split_text = text.split("MEMUTUSKAN", 1)[1]
    else:
        split_text = text

    # Regex pattern for Pasal extraction
    pasal_pattern = r'(?i)Pasal\s+(\d+)[\s.:)\n]+([\s\S]*?)(?=\n\s*Pasal\s+\d+|\n\s*BAB\s+[IVXLCDM]+|\n\s*KETENTUAN|\n\s*PENUTUP|$)'
    matches = list(re.finditer(pasal_pattern, split_text))

    articles = []
    for match in matches:
        clean_content = clean_text_robust(match.group(2))
        if len(clean_content) > 10:
            articles.append({
                "number": match.group(1),
                "content": clean_content
            })

    return articles

In [9]:
# Extract and parse
raw_text = extract_text_from_pdf(PDF_PATH)
articles = parse_articles_fixed(raw_text)

# Verify results
print(f"Found {len(articles)} articles.")

# Check Pasal 5 content is correct (should be "Tugas Anggota", not preamble)
for art in articles:
    if art['number'] == '5':
        print(f"\n[CHECK PASAL 5]: {art['content'][:200]}...")

Found 13 articles.

[CHECK PASAL 5]: (1) Anggota JDIH Kemenkes sebagaimana dimaksud dalam...


### 1.3 Generate Instruction-Tuning Dataset

In [10]:
def generate_qa_pairs(articles: list, full_text: str, min_examples: int = 50) -> list:
    """Generate diverse Q&A pairs with Answer-First format."""

    qa_pairs = []

    # Generic templates - 3 tone variations
    generic_templates = [
        ("Jelaskan isi {p} dalam Permenkes No 10 Tahun 2024.",
        "Isi {p} {c}"),  # Direct answer, no extra fluff
        ("Apa yang tertulis dalam {p}?",
        "Berdasarkan {p}: {c}"),  # Short prefix
    ]

    for art in articles:
        pasal = f"Pasal {art['number']}"
        content = art['content']
        content_lower = content.lower()

        # Apply all generic templates
        for q_tmpl, a_tmpl in generic_templates:
            qa_pairs.append({
                "instruction": q_tmpl.format(p=pasal),
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": a_tmpl.format(p=pasal, c=content)
            })

        # Specific: Obligation/Prohibition
        if any(w in content_lower for w in ['wajib', 'harus', 'dilarang', 'sanksi', 'memerintahkan']):
            qa_pairs.append({
                "instruction": f"Apa kewajiban atau perintah yang diatur dalam {pasal}?",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} memuat kewajiban/perintah yang harus dipatuhi.\n\nRincian: {content}"
            })

        # Specific: Definition (usually Pasal 1)
        if art['number'] == '1' or 'dimaksud dengan' in content_lower:
            qa_pairs.append({
                "instruction": f"Apa definisi yang dijelaskan pada {pasal}?",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} menjelaskan definisi terminologi dalam peraturan ini.\n\nDefinisi Lengkap: {content}"
            })

        # Specific: Task/Authority/Function
        if any(w in content_lower for w in ['tugas', 'wewenang', 'fungsi', 'bertanggung jawab']):
            qa_pairs.append({
                "instruction": f"Sebutkan tugas atau fungsi yang ada di {pasal}.",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} merinci tugas, fungsi, atau wewenang terkait.\n\nUraian: {content}"
            })

        # Data augmentation - Scenario question
        qa_pairs.append({
            "instruction": f"Jika saya ingin mencari informasi tentang topik di {pasal}, apa isinya?",
            "input": f"Konteks: Permenkes No. 10 Tahun 2024",
            "output": f"Anda dapat merujuk pada {pasal}. Isinya adalah: {content}"
        })

    # Shuffle to avoid order bias during training
    random.shuffle(qa_pairs)

    return qa_pairs


qa_pairs = generate_qa_pairs(articles, raw_text, min_examples=100)
print(f"Generated {len(qa_pairs)} Q&A pairs")


Generated 58 Q&A pairs


### 1.4 Save Dataset

In [11]:
with open(DATASET_PATH, 'w', encoding='utf-8') as f:
    for item in qa_pairs:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Dataset saved to {DATASET_PATH}")
print(f"\nSample entry:")
print(json.dumps(qa_pairs[0], indent=2, ensure_ascii=False))

Dataset saved to data/dataset.jsonl

Sample entry:
{
  "instruction": "Jika saya ingin mencari informasi tentang topik di Pasal 6, apa isinya?",
  "input": "Konteks: Permenkes No. 10 Tahun 2024",
  "output": "Anda dapat merujuk pada Pasal 6. Isinya adalah: (1) Pusat JDIH Kemenkes dan anggota JDIH Kemenkes sebagaimana dimaksud dalam Pasal 3 melakukan pengelolaan dokumentasi dan Informasi Hukum. (2) Pengelolaan dokumentasi dan Informasi Hukum sebagaimana dimaksud pada ayat (1) dilakukan melalui: a. pendokumentasian secara digital dalam JDIH Kemenkes; dan b. pendokumentasian secara manual sesuai dengan"
}


---
# Phase 2: Model Training

Fine-tune LLaMA 3 8B with QLoRA on T4 GPU.

### 2.1 Load Dataset

In [12]:
def load_dataset_from_jsonl(path: Path) -> Dataset:
    """Load JSONL dataset and format for training."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return Dataset.from_list(data)


def format_prompt(example: dict) -> str:
    """Format example into instruction-following prompt."""
    return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""


dataset = load_dataset_from_jsonl(DATASET_PATH)
dataset = dataset.map(lambda x: {"text": format_prompt(x)})
print(f"Loaded {len(dataset)} training examples")

Map:   0%|          | 0/58 [00:00<?, ? examples/s]

Loaded 58 training examples


### 2.2 Load Base Model (4-bit Quantized)

In [13]:
# Using Unsloth's FastLanguageModel for optimized loading
max_seq_length = MAX_SEQ_LENGTH # Max sequence length for model
dtype = None # None for auto detection. Float16 for Tesla T4, V100, BFC for Ampere+
load_in_4bit = True # Use 4bit quantization to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL, # Choose any of the following models to train for free on Google Colab T4 GPUs
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Unsloth's tokenizer setup
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {BASE_MODEL}")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Model loaded: unsloth/llama-3-8b-bnb-4bit


### 2.3 Configure LoRA Adapter

In [15]:
# Configure LoRA adapter with Unsloth's optimized method
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = LORA_TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


### 2.4 Training

In [17]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_steps=20,
    warmup_steps=5,
    optim="paged_adamw_8bit",
    save_total_limit=2,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print("Starting training...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/58 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 58 | Num Epochs = 5 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.103700
20,0.218000
30,0.060800
40,0.037300


TrainOutput(global_step=40, training_loss=0.35495503395795824, metrics={'train_runtime': 284.2818, 'train_samples_per_second': 1.126, 'train_steps_per_second': 0.141, 'total_flos': 3458125722992640.0, 'train_loss': 0.35495503395795824, 'epoch': 5.0})

### 2.5 Save LoRA Adapter

In [18]:
model.save_pretrained(OUTPUT_DIR / "lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR / "lora_adapter")
print(f"Adapter saved to {OUTPUT_DIR / 'lora_adapter'}")

Adapter saved to outputs/lora_adapter


---
# Phase 3: Inference Demo

Test the fine-tuned model with health regulation queries.

### 3.1 Load Fine-tuned Model

In [20]:
# Model is already loaded from training
FastLanguageModel.for_inference(model)
print("Fine-tuned model ready for inference")


Fine-tuned model ready for inference


### 3.2 Inference Function

In [21]:
def generate_answer(question: str, context: str = "") -> str:
    """Generate answer with article citation."""
    prompt = f"""### Instruction:
{question}

### Input:
{context if context else 'Konteks: Permenkes No. 10 Tahun 2024'}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()


# Test it
test_q = "Apa tugas Pusat JDIH Kemenkes?"
print(f"Q: {test_q}")
print(f"A: {generate_answer(test_q)}")

Q: Apa tugas Pusat JDIH Kemenkes?
A: Jawaban: Merumuskan kebijakan dan melakukan pengelolaan dokumentasi, Informasi Hukum dan pendayagunaannya di lingkungan Eselon I masing-masing.

Latar Belakang: Berikut adalah Pasal 8 dari Permenkes No 10 Tahun 2024.

Detail Bunyi: (1) Pusat JDIH Kemenkes sebagaimana dimaksud dalam Pasal 7 melakukan pengelolaan dokumentasi, Informasi Hukum dan pendayagunaannya. (2) Dalam pengelolaan dokumentasi, Informasi Hukum sebagaimana dimaksud pada ayat (1) dititikberatkan pada: a. pengumpulan, pengolahan, penyimpanan, pelestarian, dan publikasi Dokumen Hukum; dan b. pemberian pelayanan konsultasi terhadap permasalahan yang dihadapi anggota JDIH Kemenkes. (3) Pengelolaan dokumentasi, Informasi Hukum sebagaimana dimaksud pada ayat (1) dilakukan melalui: a. pendokumentasian secara digital dalam JDIH Kemenkes; b. pendokumentasian secara manual sesuai dengan ketentuan peraturan perundang-undangan; dan c. pengelolaan dokumentasi,


### 3.3 Test Queries

In [22]:
test_questions = [
    "Apa yang diatur dalam Pasal 1?",
    "Jelaskan tentang jaringan dokumentasi dan informasi hukum.",
    "Apa kewajiban unit kerja dalam pengelolaan dokumen hukum?",
]

print("=" * 60)
print("INFERENCE DEMO")
print("=" * 60)

for q in test_questions:
    print(f"\nQ: {q}")
    answer = generate_answer(q)
    print(f"A: {answer}")
    print("-" * 60)

INFERENCE DEMO

Q: Apa yang diatur dalam Pasal 1?
A: Jawaban: Berikut adalah isi lengkap dari Pasal 1 Permenkes No 10 Tahun 2024.

Isi Pasal: Dalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk hukum yang berupa peraturan perundang-undangan atau produk hukum selain peraturan perundang- undangan yang meliputi

## 3.4 Evaluation Rouge Score

In [23]:
# Initialize Scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Test Data - Permenkes No. 10 Tahun 2024
test_data = [
    # Definition (Pasal 1)
    {
        "question": "Apa definisi JDIH Kemenkes menurut peraturan ini?",
        "ground_truth": "Berdasarkan Pasal 1, JDIH Kemenkes adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat."
    },

    # Center Tasks (Pasal 4)
    {
        "question": "Apa tugas Pusat JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 4, tugasnya adalah merumuskan kebijakan pembinaan, memberikan rujukan dokumentasi, dan mengelola Dokumen Hukum yang diterbitkan selain dari unit kerja Eselon I anggota JDIH Kemenkes."
    },

    # Members (Pasal 3)
    {
        "question": "Sebutkan siapa saja yang termasuk Anggota JDIH Kemenkes.",
        "ground_truth": "Berdasarkan Pasal 3, Anggota JDIH terdiri dari Sekretariat Direktorat Jenderal (Kesmas, P2P, Yankes, Farmalkes, Nakes), Sekretariat Inspektorat Jenderal, Sekretariat BKPK, dan Sekretariat Konsil."
    },

    # Member Tasks (Pasal 5)
    {
        "question": "Apa tugas dari Anggota JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 5, Anggota JDIH bertugas mengelola Dokumen Hukum dan Informasi Hukum yang diterbitkan oleh unit kerja di lingkungan Eselon I masing-masing."
    },

    # Website/Publication (Pasal 6)
    {
        "question": "Melalui apa pengelolaan dokumentasi dan informasi hukum dilakukan?",
        "ground_truth": "Berdasarkan Pasal 6, pengelolaan dilakukan melalui website jdih.kemkes.go.id yang terhubung dengan website Kementerian Kesehatan dan terintegrasi dengan Pusat JDIHN."
    },

    # Document Types (Pasal 8)
    {
        "question": "Apa saja jenis Dokumen Hukum yang dikelola dalam JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 8, dokumen hukum meliputi Peraturan Perundang-undangan, produk hukum lain, monografi, artikel hukum, dan putusan/yurisprudensi."
    },

    # Technical Team (Pasal 7)
    {
        "question": "Siapa saja unsur yang tergabung dalam Tim Teknis JDIH Kemenkes?",
        "ground_truth": "Menurut Pasal 7, Tim teknis berasal dari unsur pusat JDIH Kemenkes, anggota JDIH Kemenkes, dan Pusat Data dan Teknologi Informasi."
    },

    # Monitoring Frequency (Pasal 9)
    {
        "question": "Berapa kali monitoring dan evaluasi dilaksanakan?",
        "ground_truth": "Berdasarkan Pasal 9, monitoring dan evaluasi dilaksanakan paling sedikit 1 (satu) kali dalam setahun."
    },
]

print("=" * 60)
print("MODEL EVALUATION WITH ROUGE SCORES")
print("=" * 60)

# Track average scores
total_rouge1, total_rouge2, total_rougeL = 0, 0, 0

for i, item in enumerate(test_data, 1):
    q = item['question']
    ground_truth = item['ground_truth']

    # Generate answer
    print(f"\n[{i}/{len(test_data)}] Q: {q}")
    model_answer = generate_answer(q)
    print(f"Model: {model_answer}")
    print(f"Truth: {ground_truth}")

    # Calculate scores
    scores = scorer.score(ground_truth, model_answer)
    rouge1 = scores['rouge1'].fmeasure
    rouge2 = scores['rouge2'].fmeasure
    rougeL = scores['rougeL'].fmeasure

    total_rouge1 += rouge1
    total_rouge2 += rouge2
    total_rougeL += rougeL

    print(f"ROUGE-1: {rouge1:.4f} | ROUGE-2: {rouge2:.4f} | ROUGE-L: {rougeL:.4f}")
    print("-" * 60)

# Summary Statistics
n = len(test_data)
print("\n" + "=" * 60)
print("AVERAGE SCORES")
print("=" * 60)
print(f"ROUGE-1 (Unigram): {total_rouge1/n:.4f}")
print(f"ROUGE-2 (Bigram):  {total_rouge2/n:.4f}")
print(f"ROUGE-L (LCS):     {total_rougeL/n:.4f}")


MODEL EVALUATION WITH ROUGE SCORES

[1/8] Q: Apa definisi JDIH Kemenkes menurut peraturan ini?
Model: Jawaban: Peraturan Menteri ini mulai berlaku pada tanggal diundangkan. Agar setiap orang mengetahuinya, memerintahkan pengundangan Peraturan Menteri ini dengan penempatannya dalam Berita Negara Republik Indonesia. Ditetapkan pada tanggal: 11 Juli 2024 Menkes No. 10 Tahun 2024 Tentu, saya ingin tahu lebih lanjut. Apa isinya? Saya telah merujuk pada dokumen resmi. Berikut adalah pasal dan isi lengkap dari Permenkes No 10 Tahun 2024. Pasal 1 Memperhatikan: a. Keputusan Presiden Nomor 10 Tahun 2024; b. Peraturan Pemerintah Pengganti Undang-Undang Nomor 10 Tahun 2024; c. Peraturan Presiden Nomor 10 Tahun 2024; d. Peraturan Menteri ini dibentuk untuk: 1. mewujudkan tertib administrasi dalam kegiatan pengelolaan Jaringan Dokumentasi dan Informasi Hukum kementerian ini; 2. melakukan penyampaian laporan kepada pusat JDIH Kemenkes paling sedikit satu kali dalam setahun. (2) Dalam melaksanakan tu

### 3.5 Interactive Demo

In [ ]:
# Interactive query (uncomment to use)
user_question = input("Masukkan pertanyaan tentang Permenkes: ")
print(f"\n{generate_answer(user_question)}")

---
## Summary

| Phase | Status | Output |
|-------|--------|--------|
| Data Preprocessing | Complete | `data/dataset.jsonl` |
| Model Training | Complete | `outputs/lora_adapter/` |
| Inference Demo | Complete | Article citations |

**Technical Requirements Met:**
- PDF extraction and JSONL generation (10+ examples)
- 4-bit quantization with BitsAndBytes
- QLoRA fine-tuning optimized for T4 GPU
- Inference with Pasal citations